In [0]:
import importlib
import src.pipeline as pipeline

importlib.reload(pipeline)

executar_pipeline = pipeline.executar_pipeline

In [0]:
resultados = executar_pipeline(
    spark,
    "workspace.default.manufacturing_defect_dataset"
)

In [0]:
display(resultados["tabela_resumo"])

## 1. Carregamento dos dados
Nesta etapa, carregamos o dataset original armazenado no Databricks. O objetivo é estabelecer a base inicial da análise e verificar se a tabela está disponível para processamento.

In [0]:
from pyspark.sql import functions as F

# carrega a tabela original do Databricks
df = spark.read.table("workspace.default.manufacturing_defect_dataset")

# exibe a estrutura da tabela
df.printSchema() 

# contagem de registros
print(f"Total de registros: {df.count()}")

# exibe os primeiros registros
display(df.limit(20))

## 2. Inspeção inicial
### Auditoria de qualidade dos dados
Nesta etapa, avaliamos a integridade da estrutura da base, verificando valores nulos, duplicados e tipos de dados.
A auditoria tem o objetivo de identificar problemas que possam comprometer as análises posteriores.

In [0]:
# valores nulos por coluna
nulos = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(nulos)

# total de registros
total_linhas = df.count()

# total de registros distintos
total_distintos = df.distinct().count()

# duplicados
duplicadas = total_linhas - total_distintos

print(f"Total de registros: {total_linhas}")
print(f"Total de registros distintos: {total_distintos}")
print(f"Total de duplicatas: {duplicadas}")


## Validando Regras de Negócio
Validação se os valores registrados fazem sentido dentro das regras que conhecemos sobre o processo, pois além de verificar se um número é negativo precisamos entender o significado da variável. 

Exemplo:
- Volume de produção negativo não faz sentido.
- Custo de produção negativo não faz sentido.
- Downtime percentual acima de 100% não faz sentido se representa percentual do tempo.
- Uma taxa de defeitos acima de 100% seria suspeita.

In [0]:

# verifica regras de negócio
regras_invalidas = df.filter(
    (F.col('MaintenanceHours') < 0) |
    (F.col('ProductionVolume') <= 0) |
    (F.col('ProductionCost') < 0) | 
    (F.col('DowntimePercentage') < 0) | 
    (F.col('DowntimePercentage') > 100) | 
    (F.col('EnergyEfficiency') <= 0) 
    )

print(f"Quantidade de registros inválidos: {regras_invalidas.count()}")
display(regras_invalidas)

## 3. Preparação dos dados
Nesta etapa, serão criados indicadores para apoiar as análises de qualidade, custos e eficiência operacional.
Os indicadores serão calculados a partir das variáveis originais, sem remover registros apenas por apresentarem valores extremos.

Após ter uma auditoria inicial, iremos criar a camada Silver e o próximo passo é a base analítica:
- Preservar todos os registros válidos
- Não remover outliers automaticamente
- Criar indicadores para a análise

In [0]:
# cria a base analítica inicial
df_silver = df 

# cria coluna de custo por unidade produzida
df_silver = df_silver.withColumn(
    "UnitCost",
    F.when(
        F.col("ProductionVolume") > 0,
        F.col("ProductionCost") / F.col("ProductionVolume")
    )
)

# cria coluna de consumo energético por unidade produzida
df_silver = df_silver.withColumn(
    "EnergyPerUnit",
    F.when(
        F.col("ProductionVolume") > 0,
        F.col("EnergyConsumption") / F.col("ProductionVolume")
    )
)

# exibe as colunas criadas
display(
    df_silver.select(
        "ProductionVolume",
        "ProductionCost",
        "UnitCost",
        "EnergyConsumption",
        "EnergyPerUnit"
    ).limit(10)
)

### Explicação para a criação dessas colunas
- UnitCost: permite comparar custos relativos ao volume de produção.
- EnergyPerUnit: permite comparar consumo energético relativo ao volume produzido. 

Essas colunas ajudam porque comparar custos totais ou o consumo total entre registros de volumes diferentes pode gerar interpretações equivocadas. 
Exemplo: uma operação que produz 1.000 unidades pode consumir mais energia que outra que produz 100 unidades, mas não significa necessariamente que é menos eficiente. Ter esse dado unitário ajuda a investigar essa diferença. 

## 4. Salvar a camada Silver 
Nesta etapa, iremos salvar os dados tratados como uma tabela Delta para poder reutilizar em outras análises. 
Isso serve para reutilizar os dados sem repetir todas as transformações, poder consultar a base com SQL, conectar o Power BI na camada analítica e também manter uma estrutura organizada no projeto.

In [0]:
# cria nome da tabela
tabela_silver = "workspace.default.manufacturing_defects_silver"

# salva a camada silver como tabela delta
(
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_silver)
)
# "overwrite" substitui a tabela de destino caso ela já exista
print(f"Tabela salva: {tabela_silver}")

In [0]:
# faz a leitura novamente da tabela para conferir se foi salva
df_silver_confere = spark.read.table("workspace.default.manufacturing_defects_silver")

# confere a quantidade de registros
print(f"Registros na camada Silver: {df_silver_confere.count()}")

# confere a estrutura
df_silver_confere.printSchema()

# exibe os dados em formato de tabela
display(df_silver_confere.limit(15))

## 5. Detecção de outliers no custo unitário

Calcula o custo unitário e usa o intervalo interquartil (IQR) para identificar e filtrar outliers.

In [0]:
# Calcula os quartis do custo unitário
quartis = df_silver.agg(
    F.expr('percentile(UnitCost, 0.25)').alias('q1'),
    F.expr('percentile(UnitCost, 0.75)').alias('q3')
).collect()[0]
q1 = quartis['q1']
q3 = quartis['q3']
iqr = q3 - q1

# Define os limites inferior e superior para outliers
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")

print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")

df_silver = df_silver.withColumn(
    "IsUnitCostOutlier",
    (F.col("UnitCost") < limite_inferior) |
    (F.col("UnitCost") > limite_superior)
)

df_silver.groupBy("IsUnitCostOutlier").count().show()

Como o limite inferior ficou um valor negativo, significa que nenhum custo unitário positivo seria considerado outlier pela regra do inferior. Com isso, entende-se que os 284 registros que foram identificados como 'true',  estão acima do limite superior de 72.

Porém, ter registros que ultrapassam o limite de 72 não significa necessariamente que esteja errado. Pode ter acontecido do volume produzido ser baixo, o custo tenha sido elevado, ou ainda exista uma situação operacional para investigar.

Por isso, o próximo passo é analisar se os registros com custo unitário extremo apresentam indicadores diferentes de qualidade, manutenção e eficiência. 

In [0]:
df_silver.groupBy("IsUnitCostOutlier").agg(
    F.count("*").alias("TotalRegistros"),
    F.avg("UnitCost").alias("CustoUnitarioMedio"),
    F.avg("ProductionCost").alias("CustoProducaoMedio"),
    F.avg("QualityScore").alias("QualidadeMedia"),
    F.avg("DefectRate").alias("TaxaDefeitoMedia"),
    F.avg("MaintenanceHours").alias("HorasManutencaoMedia"),
    F.avg("EnergyPerUnit").alias("EnergiaPorUnidadeMedia"),
    F.avg("DefectStatus").alias("MediaDefectStatus")).show()

A análise dos outliers de custo unitário mostra uma associação entre custos unitários elevados e maior consumo de energia por unidade, enquanto que as médias de qualidade e taxa de defeitos apresentaram diferenças pequenas entre os grupos. 

Os resultados não permitem afirmar relações de causa e efeito. Ainda não serão excluídos esses 284 registros, pois eles podem representar situações operacionais relevantes, e não erros nos dados.

In [0]:
# verifica se existe equilibrio entre baixo e alto nível de defeitos
df_silver.groupBy("DefectStatus").agg(
    F.count("*").alias("TotalRegistros"),
    F.round(F.count("*") / df_silver.count() * 100, 2).alias("Percentual")
).orderBy("DefectStatus").show()

A análise da distribuição da coluna 'DefectStatus' identificou que existe um desequilíbrio entre as classificações, sendo que 84,04% dos registros apresentam alto nível de defeitos, enquanto que 15,96% apresentam baixo nível. Isso deve ser considerado na interpretação das análises.

In [0]:
df_silver.groupBy("DefectStatus").agg(
    F.count("*").alias("TotalRegistros"),
    F.round(F.avg("DefectRate"), 2).alias("TaxaDefeitoMedia"),
    F.round(F.avg("QualityScore"), 2).alias("QualidadeMedia"),
    F.round(F.avg("MaintenanceHours"), 2).alias("HorasManutencaoMedia"),
    F.round(F.avg("ProductionVolume"), 2).alias("VolumeProducaoMedio"),
    F.round(F.avg("UnitCost"), 2).alias("CustoUnitarioMedio"),
    F.round(F.avg("EnergyPerUnit"), 2).alias("EnergiaPorUnidadeMedia")
).orderBy("DefectStatus").show()

In [0]:
import matplotlib.pyplot as plt

# agrupa os dados pela classificação de defeitos
df_grafico = df_silver.groupBy("DefectStatus").agg(F.round(F.avg("DefectRate"), 2).alias("TaxaDefeitoMedia")).orderBy("DefectStatus")

# converte o resultado do PySpark para Pandas
df_grafico_pd = df_grafico.toPandas()

# exibe a tabela que será utilizada no gráfico
display(df_grafico_pd)

In [0]:
# variaveis para definir os dados do gráfico
categorias = ["Baixo nível de defeitos", "Alto nível de defeitos"]
valores = df_grafico_pd["TaxaDefeitoMedia"]

# cria o gráfico
plt.figure(figsize=(9, 5))
plt.bar(categorias, valores)

# personaliza o gráfico
plt.title("Registros com alto nível de defeitos apresentam maior taxa média")
plt.xlabel("Classificação de defeitos")
plt.ylabel("Taxa média de defeitos")

# exibe os valores acima das barras
for i, valor in enumerate(valores):
    plt.text(i, valor + 0.03, str(valor), ha="center")

plt.tight_layout()
plt.show()

### Interpretação do gráfico - Taxa média de defeitos

Demonstra que os registros classificados com alto nível de defeitos apresentam uma taxa média de defeitos de 2,89 enquanto os registros com baixo nível apresentam uma média de 2,01.

Essa diferença mostra uma variação entre os dois grupos, indicando que a classificação de alto nível de defeitos está associada a uma maior taxa média de defeitos. Esse resultado é coerente com a classificação utilizada no dataset e ajuda a caracterizar os diferentes níveis de qualidade da produção.

In [0]:
# agrupa os dados pela classificação de defeitos
df_manutencao = df_silver.groupBy("DefectStatus").agg(
    F.round(F.avg("MaintenanceHours"), 2).alias("HorasManutencaoMedia")
).orderBy("DefectStatus")

# converte o resultado para pandas
df_manutencao_pd = df_manutencao.toPandas()

# exibe a tabela
display(df_manutencao_pd)

In [0]:
# define os dados do gráfico
categorias = ["Baixo nível de defeitos", "Alto nível de defeitos"]
valores = df_manutencao_pd["HorasManutencaoMedia"]

# cria o gráfico
plt.figure(figsize=(9, 5))
plt.bar(categorias, valores)

# personaliza o gráfico
plt.title("Registros com alto nível de defeitos apresentam mais horas de manutenção")
plt.xlabel("Classificação de defeitos")
plt.ylabel("Média de horas de manutenção")

# exibe os valores acima das barras
for i, valor in enumerate(valores):
    plt.text(i, valor + 0.15, str(valor), ha="center")

plt.tight_layout()
plt.show()

### Interpretação do gráfico - Horas médias de manutenção
Apresenta uma diferença nas horas médias de manutenção entre os dois grupos. Os registros com alto nível de defeitos apresentam uma média de 12,37 horas de manutenção, enquanto os registros com baixo nível apresentam uma média de 6,79 horas.

Isso representa aproximadamente 82% mais horas de manutenção no grupo com alto nível de defeitos. Esse resultado indica uma associação entre maiores níveis de defeitos e maior necessidade de manutenção nos registros analisados. Entretanto, não é possível afirmar que o aumento das horas de manutenção causa o aumento dos defeitos, pois a análise é descritiva.